# IHC Binary Evaluation — ElSherief et al. (2021)

Evaluates baseline and RAC models on the IHC test split (explicit hate removed) and compares macro F1 against published results.

## 1. Imports

In [1]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import faiss
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve()))
from retriever import retrieve_top_k_above_threshold

## 2. Configuration

In [2]:
# Define paths
ROOT_DIR        = Path('../..')
WEIGHTS_DIR     = ROOT_DIR / 'weigths' / 'weights_baseline'
WEIGHTS_RAC_DIR = ROOT_DIR / 'weigths' / 'weights_rac_best_hyperparameters'
INDEX_DIR       = ROOT_DIR / 'corpus' / 'index'

# Model config
MAX_LENGTH = 256
BATCH_SIZE = 32

# Retrieval config
K             = 5
THRESHOLD     = 0.4
SBERT_HF_ID   = 'sentence-transformers/all-mpnet-base-v2'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## 3. Load & Filter IHC

Same split and filter as training: 90/10 train/test (seed=42), explicit hate removed, binary labels.

In [3]:
from data_loaders import load_ihc_implicit_only

test_ihc = load_ihc_implicit_only(seed=42)

print(f'IHC test — after removing explicit hate: {len(test_ihc):,}')
print(f'  Non-hate: {sum(1 for x in test_ihc if x["label"] == 0):,}')
print(f'  Implicit hate: {sum(1 for x in test_ihc if x["label"] == 1):,}')

README.md:   0%|          | 0.00/792 [00:00<?, ?B/s]

implicit_hate_v1_stg1_posts.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/21480 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2148 [00:00<?, ? examples/s]

Map:   0%|          | 0/2028 [00:00<?, ? examples/s]

IHC test — after removing explicit hate: 2,028
  Non-hate: 1,330
  Implicit hate: 698


## 4. Model Registry

List of all models to evaluate — baseline and RAC variants.

In [4]:
# Baseline models
BASELINE_MODELS = [
    {'path': WEIGHTS_DIR / 'bert' / 'IHC', 'label': 'BERT (baseline)'},
    {'path': WEIGHTS_DIR / 'hatebert' / 'IHC',          'label': 'HateBERT (baseline)'},
    {'path': WEIGHTS_DIR / 'roberta' / 'IHC',      'label': 'RoBERTa (baseline)'},
]

RAG_MODELS = [
    {
        'path':             WEIGHTS_RAC_DIR / 'bert' / 'sbert' / 'full' / 'IHC',
        'label':            'BERT (RAC sbert/full)',
        'retriever_hf_id':  SBERT_HF_ID,
        'index_type':       'full',
    },
    {
        'path':             WEIGHTS_RAC_DIR / 'bert' / 'sbert' / 'training' / 'IHC',
        'label':            'BERT (RAG sbert/example)',
        'retriever_hf_id':  SBERT_HF_ID,
        'index_type':       'example',
    },
    {
        'path':             WEIGHTS_RAC_DIR / 'bert' / 'sbert' / 'documents' / 'IHC',
        'label':            'BERT (RAG sbert/knowledge)',
        'retriever_hf_id':  SBERT_HF_ID,
        'index_type':       'knowledge',
    },
    {
        'path':             WEIGHTS_RAC_DIR / 'roberta' / 'sbert' / 'full' / 'IHC',
        'label':            'RoBERTa (RAC sbert/full)',
        'retriever_hf_id':  SBERT_HF_ID,
        'index_type':       'full',
    },
    {
        'path':             WEIGHTS_RAC_DIR / 'roberta' / 'sbert' / 'training' / 'IHC',
        'label':            'RoBERTa (RAG sbert/example)',
        'retriever_hf_id':  SBERT_HF_ID,
        'index_type':       'example',
    },
    {
        'path':             WEIGHTS_RAC_DIR / 'roberta' / 'sbert' / 'documents' / 'IHC',
        'label':            'RoBERTa (RAG sbert/knowledge)',
        'retriever_hf_id':  SBERT_HF_ID,
        'index_type':       'knowledge',
    },
]

# Combine both lists
all_models = BASELINE_MODELS + RAG_MODELS

print(f"{'Model':<35} {'Type':<10} Weights?")
print('-' * 58)
for m in all_models:
    has  = (m['path'] / 'model.safetensors').exists() or (m['path'] / 'pytorch_model.bin').exists()
    kind = 'RAG' if 'retriever_hf_id' in m else 'baseline'
    print(f"{m['label']:<35} {kind:<10} {'✓' if has else '✗  (missing)'}")

Model                               Type       Weights?
----------------------------------------------------------
BERT (baseline)                     baseline   ✓
HateBERT (baseline)                 baseline   ✓
RoBERTa (baseline)                  baseline   ✓
BERT (RAC sbert/full)               RAG        ✓
BERT (RAG sbert/example)            RAG        ✗  (missing)
BERT (RAG sbert/knowledge)          RAG        ✗  (missing)
RoBERTa (RAC sbert/full)            RAG        ✓
RoBERTa (RAG sbert/example)         RAG        ✗  (missing)
RoBERTa (RAG sbert/knowledge)       RAG        ✗  (missing)


## 5. Helpers

In [5]:
from training_utils import compute_metrics, tokenize_augmented, tokenize_plain


# augment_test: retrieve neighbors for a test split (no self-exclusion)
def augment_test(hf_dataset, ret_model, ret_tokenizer, ret_index, ret_documents):
    records = []
    for example in tqdm(hf_dataset, desc='augmenting'):
        neighbors = retrieve_top_k_above_threshold(
            example['post'], THRESHOLD, ret_model, ret_tokenizer,
            ret_index, ret_documents, chunk_id=None, k=K, use_mean_pool=True,
        )
        records.append({
            'query':     example['post'],
            'neighbors': [text for text, _ in neighbors],
            'label':     example['label'],
        })
    return records

## 6. Evaluation Loop

In [ ]:
results = {}

# Set up a minimal Trainer just for inference
eval_args = TrainingArguments(
    output_dir='./tmp_eval',
    per_device_eval_batch_size=BATCH_SIZE,
    report_to='none',
)

# Evaluate each model
for entry in all_models:
    has_weights = (entry['path'] / 'model.safetensors').exists() or (entry['path'] / 'pytorch_model.bin').exists()
    if not has_weights:
        print(f"[skip] {entry['label']} — no weights on disk")
        continue

    print(f"\n{'='*55}")
    print(f"{entry['label']}")
    print(f"{'='*55}")

    is_rag = 'retriever_hf_id' in entry

    if is_rag:
        # ── Retrieval augmentation via sbert ──────────────────────
        print('Loading sbert retriever and augmenting test set...')
        ret_tokenizer = AutoTokenizer.from_pretrained(entry['retriever_hf_id'])
        ret_model     = AutoModel.from_pretrained(entry['retriever_hf_id']).eval().to(device)
        ret_index     = faiss.read_index(
            str(INDEX_DIR / f"vdb_{entry['index_type']}.faiss")
        )
        with open(INDEX_DIR / f"lookup_{entry['index_type']}.json") as f:
            ret_documents = json.load(f)

        aug_records = augment_test(test_ihc, ret_model, ret_tokenizer, ret_index, ret_documents)

        del ret_model, ret_tokenizer
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        tokenizer = AutoTokenizer.from_pretrained(entry['path'])
        tok_test  = tokenize_augmented(aug_records, tokenizer)

    else:
        # ── Plain text for baseline models ────────────────────────
        tokenizer = AutoTokenizer.from_pretrained(entry['path'])
        tok_test  = tokenize_plain(test_ihc, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(entry['path'])

    trainer = Trainer(
        model=model,
        args=eval_args,
        compute_metrics=compute_metrics,
    )

    preds_out = trainer.predict(tok_test)
    preds  = np.argmax(preds_out.predictions, axis=-1)
    labels = list(test_ihc['label'])

    print(classification_report(labels, preds, target_names=['Non-hate', 'Implicit hate']))

    results[entry['label']] = {
        'macro_f1': f1_score(labels, preds, average='macro',  zero_division=0),
        'macro_p':  precision_score(labels, preds, average='macro', zero_division=0),
        'macro_r':  recall_score(labels, preds, average='macro',    zero_division=0),
    }

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

## 7. Paper Results

Published results from Table 3 of ElSherief et al. (2021) for comparison.

In [7]:
PAPER_RESULTS = {
    'SVM n-grams (paper)':     {'macro_f1': 0.644, 'macro_p': 0.614, 'macro_r': 0.677},
    'BERT (paper)': {'macro_f1': 0.689, 'macro_p': 0.721, 'macro_r': 0.660},
    'BERT + Aug (paper)':  {'macro_f1': 0.704, 'macro_p': 0.678, 'macro_r': 0.732},
}

## 8. Results — Comparison Table

Our models vs. published baselines.

In [8]:
all_results = dict(results)
for label, vals in PAPER_RESULTS.items():
    if vals['macro_f1'] is not None:
        all_results[label] = vals

df = pd.DataFrame({
    label: {'Macro F1': v['macro_f1'], 'Macro Precision': v['macro_p'], 'Macro Recall': v['macro_r']}
    for label, v in all_results.items()
}).T
df.index.name = 'Model'

display(
    df.style
    .format('{:.3f}')
    .highlight_max(axis=0, props='font-weight: bold; background-color: #d4f1d4')
    .set_caption('IHC binary evaluation (implicit hate only) — ElSherief et al. (2021) comparison')
)

,Macro F1,Macro Precision,Macro Recall
Model,,,
BERT (baseline),0.778,0.781,0.776
HateBERT (baseline),0.780,0.785,0.777
RoBERTa (baseline),0.789,0.793,0.786
BERT (RAC sbert/full),0.767,0.767,0.767
RoBERTa (RAC sbert/full),0.795,0.799,0.791
SVM n-grams (paper),0.644,0.614,0.677
BERT (paper),0.689,0.721,0.660
BERT + Aug (paper),0.704,0.678,0.732
